# Lecture 10b examples: Conditional distributions and simulation

STAT-S681 · Simulation-Based Inference

In [1]:
import numpy as np

Code behind the lecture 10b slides: rejection sampling for the three examples.
Each example has a simulator for the unconditional distribution (step 1) and a
function that repeats it until the outcome is in $E$ (steps 2-4).

# Disease example

Simulate $(D, T)$: disease status, then a test result whose probability depends on $D$.

In [2]:
rng = np.random.default_rng(681)

def simulate_DT():
    D = rng.binomial(1, 0.01)
    p = 0.90 if D == 1 else 0.05
    T = rng.binomial(1, p)
    return {"D": D, "T": T}

Keep $D$ from the first run with a positive test, $(D,T) \in E = \{(0,1), (1,1)\}$.

In [3]:
def rejection_sample_DT():
    while True:
        DT = simulate_DT()
        if DT["T"] == 1:
            return DT["D"]

print(rejection_sample_DT())

0


Repeat to estimate $\Pr(D = 1 \mid (D,T) \in E)$; the exact value is $2/13 \approx 0.154$.

In [4]:
draws = np.array([rejection_sample_DT() for _ in range(5000)])
print(draws.mean())

0.1556


# Duration example

Simulate the cessation week $R$ from the table in Example 1.

In [5]:
rng = np.random.default_rng(681)

def simulate_R():
    probs = [0.082, 0.092, 0.099,
             0.151, 0.175, 0.086,
             0.110, 0.072, 0.041,
             0.092]
    weeks = np.arange(7, 17)
    R = rng.choice(weeks, p=probs)
    return R

Keep $R$ from the first run with $R \in E = \{14, 15, 16\}$.

In [6]:
def rejection_sample_R():
    while True:
        R = simulate_R()
        if R in [14, 15, 16]:
            return R

print(rejection_sample_R())

16


Repeat to estimate the conditional distribution of $R$; the exact values are
$0.35$, $0.20$ and $0.45$ for weeks 14, 15 and 16.

In [7]:
draws = np.array([rejection_sample_R() for _ in range(5000)])
for week in [14, 15, 16]:
    print(week, np.mean(draws == week))

14 0.351
15 0.1984
16 0.4506


# Series example

Simulate all five games ($1$ means A wins) and the series winner $W$.

In [8]:
rng = np.random.default_rng(681)

def simulate_series():
    X = rng.binomial(1, 0.5, 5)
    W = int(X.sum() >= 3)
    return {"X": X, "W": W}

Keep $W$ from the first run whose first three games are A, B, A.

In [9]:
def rejection_sample_series():
    while True:
        XW = simulate_series()
        start = list(XW["X"][:3])
        if start == [1, 0, 1]:
            return XW["W"]

print(rejection_sample_series())

1


Repeat to estimate $\Pr(W = 1 \mid \mathbf{X} \in E)$; the exact value is $3/4$.

In [10]:
draws = np.array([rejection_sample_series() for _ in range(5000)])
print(draws.mean())

0.7536
